# Machine Learning Final: Which companies would *you* apply to?
# 機械学習 期末課題：あなたが応募したい企業を予測する

In this notebook you will:

1. **Collect real job data** from Mynavi 2028 (マイナビ2028) by saving search result pages in your own browser.
2. **Extract** the company information from those pages with Beautiful Soup.
3. **Clean** the Japanese text with the Janome tokenizer.
4. **Label** each company: 1 = I'm interested, 0 = I'm not.
5. **Train** a Multinomial Naive Bayes classifier that learns your preferences.
6. **Use** the model to rank companies you haven't looked at yet.

### Why do we save pages instead of scraping the site automatically?

Mynavi's `robots.txt` (https://job.mynavi.jp/robots.txt) tells automated programs **not** to fetch any URL that contains a `?`,
which includes every search result page, and to wait **120 seconds** between requests.
Saving the pages yourself while browsing normally respects those rules,
and your Mynavi password never goes into this notebook.

Mynavi の `robots.txt` は、`?` を含む URL（検索結果ページを含む）の自動取得を禁止しています。
そのため、この課題ではブラウザで検索結果ページを自分で保存し、そのファイルを解析します。パスワードをノートブックに入力する必要はありません。

## Step 1: Save Mynavi search result pages (do this in your browser)
## ステップ1：マイナビの検索結果ページを保存する

1. Open **https://job.mynavi.jp/28/pc/** in Chrome. You can log in to your own account if you want to; it is not required.
2. Search for companies (by industry, region, keyword, internship, ...). Choose searches that include **both** companies you like and companies you don't, so the model can learn the difference.
3. On each results page press **Ctrl+S** (Windows) or **⌘+S** (Mac) and save as **"Webpage, HTML Only"** (「ウェブページ、HTMLのみ」).
4. Go to the next page of results and save again. Each page has about **30 companies**, so **7 pages ≈ 200 companies**.

⚠️ If you were logged in, the saved files may contain your name. This notebook only reads the company cards, but **do not share the HTML files** with anyone.

1. Chrome で https://job.mynavi.jp/28/pc/ を開きます（ログインは任意です）。
2. 興味のある企業とない企業の両方が含まれるように検索します。
3. 各検索結果ページで **Ctrl+S**（Mac は **⌘+S**）を押し、「ウェブページ、HTMLのみ」で保存します。
4. 次のページに進んで同じように保存します。1ページ約30社なので、7ページで約200社になります。

## Step 2: Install the Japanese tokenizer
Janome is a Japanese morphological analyzer (形態素解析エンジン): https://pypi.org/project/Janome/

In [ ]:
!pip install janome

## Step 3: Upload your saved pages
Run the cell and choose **all** the `.html` files you saved (you can select several at once). A `.zip` of the files also works.

保存した `.html` ファイルをすべて選択してアップロードしてください（複数選択可・zip も可）。

In [ ]:
import os
import shutil
import zipfile

PAGES_DIR = 'mynavi_pages'
os.makedirs(PAGES_DIR, exist_ok=True)

try:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                z.extractall(PAGES_DIR)
        else:
            shutil.move(name, os.path.join(PAGES_DIR, name))
except ImportError:
    print(f'Not running in Colab: put your saved pages in the "{PAGES_DIR}" folder.')

html_files = sorted(
    os.path.join(root, f)
    for root, _, names in os.walk(PAGES_DIR)
    for f in names
    if f.lower().endswith(('.html', '.htm'))
)
print(f'{len(html_files)} HTML file(s) ready:')
for f in html_files:
    print(' -', f)

## Step 4: Extract the companies with Beautiful Soup

Every company on a Mynavi results page is inside a `<div class="boxSearchresultEach">` box ("card").
Open one of your saved files, right-click a company name and choose **Inspect** (検証) to see it yourself.

For each card we keep the company name, its industry (業種), headquarters (本社), number of employees (従業員),
and all of the card's text. A company that appears on several pages is kept only once.

In [ ]:
import re
import pandas as pd
from bs4 import BeautifulSoup

CARD_SELECTOR = 'div.boxSearchresultEach'
CORP_LINK = re.compile(r'/search/corp(\d+)/outline\.html')
# Button and link text that appears on every card and says nothing about the company
NOISE = ['エントリー', '検討リスト登録', 'この企業の他のコース情報を見る', '説明会・面接日程']


def find_cards(soup):
    cards = soup.select(CARD_SELECTOR)
    if cards:
        return cards
    # Fallback in case Mynavi renames the class: climb up from each company link
    # until the box would contain a second company.
    cards = []
    for a in soup.find_all('a', href=CORP_LINK):
        box = a
        while box.parent is not None:
            ids = {m.group(1) for link in box.parent.find_all('a', href=CORP_LINK)
                   if (m := CORP_LINK.search(link['href']))}
            if len(ids) > 1:
                break
            box = box.parent
        if box not in cards:
            cards.append(box)
    return cards


def field(card, label):
    # Read a value such as 業種 or 本社 from the card's label list
    for li in card.select('ul.linkCtg li'):
        ttl = li.select_one('.ttl')
        if ttl and label in ttl.get_text().replace('\u3000', ''):
            ttl.extract()
            return li.get_text('', strip=True)
    return ''


records = {}
for path in html_files:
    with open(path, encoding='utf-8', errors='ignore') as f:
        soup = BeautifulSoup(f.read(), 'lxml')
    for card in find_cards(soup):
        link = card.find('a', href=CORP_LINK)
        if link is None:
            continue
        corp_id = CORP_LINK.search(link['href']).group(1)
        if corp_id in records:
            continue
        text = card.get_text(' ', strip=True).replace('\u3000', '')
        for noise in NOISE:
            text = text.replace(noise, ' ')
        records[corp_id] = {
            'corp_id': corp_id,
            'company': link.get_text(strip=True),
            'industry': field(card, '業種'),
            'headquarters': field(card, '本社'),
            'employees': field(card, '従業員'),
            'Company_info': re.sub(r'\s+', ' ', text).strip(),
        }

df = pd.DataFrame(records.values())
print(f'Found {len(df)} unique companies in {len(html_files)} file(s).')
if df.empty:
    print('No companies found. Did you save Mynavi *search result* pages as "Webpage, HTML Only"?')
else:
    # -1 means "not labeled yet"; you will change it to 1 or 0 in Step 6
    df['entry?'] = -1
    df.to_csv('2026companies.csv', index=False, encoding='utf-8')
    display(df[['company', 'industry', 'headquarters', 'employees']].head(10))

## Step 5: Clean the Japanese text

Janome splits Japanese text into words. We then remove:
- punctuation and symbols (品詞 = 記号)
- **stop words**: very common words such as これ, もの, ある that don't help tell companies apart.
  The list is written out in the cell below, so you can read it and change it.
- **Mynavi labels**: words such as 業種 and 応募締切 that are printed on *every* company card, so they tell us nothing about any one company.

**Try it:** after training in Step 7, look at the "words most linked to interested." If you see words that are just page decoration, add them to `MY_STOPWORDS` and run Steps 5 and 7 again. Does the accuracy change?

The result is one string of space-separated words per company, which is what the classifier needs.

In [ ]:
from janome.tokenizer import Tokenizer

# Japanese stop words from the stopwords-iso project:
# https://github.com/stopwords-iso/stopwords-ja
# Copyright (c) 2016 Gene Diaz. Released under the MIT License.
JAPANESE_STOPWORDS = '''
あそこ あっ あの あのかた あの人 あり あります ある あれ い いう います
いる う うち え お および おり おります か かつて から が
き ここ こちら こと この これ これら さ さらに し しかし する
ず せ せる そこ そして その その他 その後 それ それぞれ それで た
ただし たち ため たり だ だっ だれ つ て で でき できる
です では でも と という といった とき ところ として とともに とも と共に
どこ どの な ない なお なかっ ながら なく なっ など なに なら
なり なる なん に において における について にて によって により による に対して
に対する に関する の ので のみ は ば へ ほか ほとんど ほど ます
また または まで も もの ものの や よう より ら られ られる
れ れる を ん 何 及び 彼 彼女 我々 特に 私 私達
貴方 貴方方
'''.split()

# Labels that Mynavi prints on every company card. Add your own words here!
MY_STOPWORDS = ['業種', '本社', '従業', '員', '開催', '地域', '時期', '応募', '締切', '実施', '日数', '未満', '以上']

japanese_stopwords = set(JAPANESE_STOPWORDS) | set(MY_STOPWORDS)
print(f'Using {len(japanese_stopwords)} stop words.')

t = Tokenizer()


def clean_text(text):
    words = []
    for token in t.tokenize(str(text)):
        part_of_speech = token.part_of_speech.split(',')[0]
        if part_of_speech == '記号':           # symbol / punctuation
            continue
        if token.surface in japanese_stopwords:
            continue
        words.append(token.surface)
    return ' '.join(words)


df = pd.read_csv('2026companies.csv', encoding='utf-8', dtype={'corp_id': str})
df['clean_info'] = df['Company_info'].apply(clean_text)
df.to_csv('2026companies.csv', index=False, encoding='utf-8')
print(df.loc[0, 'Company_info'][:150])
print('↓')
print(df.loc[0, 'clean_info'][:150])

## Step 6: Label the companies (1 = interested, 0 = not interested)

Run the cell and answer for each company:
- **y** = yes, I would apply (1)
- **n** = no (0)
- **s** = skip this one for now
- **q** = quit and save (run the cell again later to continue where you stopped)

Your answers are saved after every company. Try to label **all** of them, and make sure you have **at least 10 "yes" and 10 "no"** answers.

**Alternative:** download `2026companies.csv` from the Files panel (📁 on the left), set the `entry?` column to 1 or 0 in Excel or Google Sheets, save it as CSV (UTF-8), and upload it again with the same name.

各企業について y（応募したい=1）/ n（応募しない=0）/ s（スキップ）/ q（保存して終了）で答えてください。

In [ ]:
import textwrap

df = pd.read_csv('2026companies.csv', encoding='utf-8', dtype={'corp_id': str})
wrapper = textwrap.TextWrapper(width=60)
todo = df.index[df['entry?'] == -1]
print(f'{len(todo)} companies left to label.')

for n, idx in enumerate(todo, start=1):
    row = df.loc[idx]
    print(f'\n[{n}/{len(todo)}] {row["company"]}  |  {row["industry"]}  |  {row["headquarters"]}  |  {row["employees"]}')
    print(wrapper.fill(row['Company_info']))
    answer = input('Would you apply to this company? (y/n/s/q): ').strip().lower()
    if answer == 'q':
        break
    if answer in ('y', 'n'):
        df.loc[idx, 'entry?'] = 1 if answer == 'y' else 0
        df.to_csv('2026companies.csv', index=False, encoding='utf-8')

labeled = df[df['entry?'] != -1]
print(f'\nLabeled: {len(labeled)}  (interested: {(labeled["entry?"] == 1).sum()}, not interested: {(labeled["entry?"] == 0).sum()})')

## Step 7: Train your Multinomial Naive Bayes classifier

Naive Bayes is a light and fast classifier that works well for text.
`CountVectorizer` turns each company's words into word counts, and the classifier learns which words show up more often in companies you chose.

We hold back 20% of your labeled companies as a **test set** the model never sees during training, and we compare its accuracy
with a **baseline** that always guesses your most common answer. A useful model has to beat the baseline.

In [ ]:
import joblib
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv('2026companies.csv', encoding='utf-8', dtype={'corp_id': str})
if 'clean_info' not in df.columns:   # CSV re-uploaded without the cleaned text
    df['clean_info'] = df['Company_info'].apply(clean_text)
df['clean_info'] = df['clean_info'].fillna('')
labeled = df[df['entry?'].isin([0, 1])]
counts = labeled['entry?'].value_counts()
if len(counts) < 2 or counts.min() < 2:
    raise ValueError('You need several companies labeled 1 AND several labeled 0. Go back to Step 6.')

train, test = train_test_split(labeled, test_size=0.2, random_state=42, stratify=labeled['entry?'])

# Japanese words can be a single character (e.g. 株, 車), so count every space-separated token
vectorizer = CountVectorizer(token_pattern=r'(?u)\S+')
X_train = vectorizer.fit_transform(train['clean_info'])
X_test = vectorizer.transform(test['clean_info'])

clf = MultinomialNB()
clf.fit(X_train, train['entry?'])
y_pred = clf.predict(X_test)

baseline = (test['entry?'] == train['entry?'].mode()[0]).mean()
print(f'Training companies: {len(train)}, test companies: {len(test)}')
print(f'Accuracy: {accuracy_score(test["entry?"], y_pred):.2f}   (baseline that always guesses your most common answer: {baseline:.2f})')
print(classification_report(test['entry?'], y_pred, target_names=['not interested (0)', 'interested (1)'], zero_division=0))

# Retrain on ALL labeled companies for the final model
clf.fit(vectorizer.fit_transform(labeled['clean_info']), labeled['entry?'])
joblib.dump(clf, 'company_classifier.pkl')
joblib.dump(vectorizer, 'company_vectorizer.pkl')

# Words that push the model most toward "interested"
words = vectorizer.get_feature_names_out()
log_ratio = clf.feature_log_prob_[1] - clf.feature_log_prob_[0]
print('Words most linked to "interested":', ', '.join(words[log_ratio.argsort()[::-1][:15]]))

## Step 8: Use your model

**8a.** Paste the description of any company (for example from a Mynavi company page) and see what the model predicts.

In [ ]:
description = input('Paste a company description: ')
vector = vectorizer.transform([clean_text(description)])
probability = clf.predict_proba(vector)[0][1]
if probability >= 0.5:
    print(f'The model predicts you WOULD be interested ({probability:.0%}).')
else:
    print(f'The model predicts you would NOT be interested ({probability:.0%} chance of interest).')

**8b.** Rank the companies you skipped or didn't label. Save more Mynavi pages, run Steps 3–5 again, and the model will tell you which new companies to look at first.

In [ ]:
df = pd.read_csv('2026companies.csv', encoding='utf-8', dtype={'corp_id': str})
if 'clean_info' not in df.columns:
    df['clean_info'] = df['Company_info'].apply(clean_text)
unlabeled = df[df['entry?'] == -1].copy()
if unlabeled.empty:
    print('Every company is labeled. Save and upload more Mynavi pages to get recommendations.')
else:
    unlabeled['interest'] = clf.predict_proba(vectorizer.transform(unlabeled['clean_info'].fillna('')))[:, 1]
    top = unlabeled.sort_values('interest', ascending=False).head(10)
    top['interest'] = top['interest'].map('{:.0%}'.format)
    top['url'] = 'https://job.mynavi.jp/28/pc/search/corp' + top['corp_id'] + '/outline.html'
    display(top[['company', 'industry', 'headquarters', 'interest', 'url']])